# Aula 6 - Treino da rede neuronal
## Losses (Slides 3-5)

In [1]:
import numpy as np
from si.neural_networks.losses import LossFunction, MeanSquaredError, BinaryCrossEntropy

# 1. a classe base é abstrata: não se pode criar um objeto dela
try:
    LossFunction()
except TypeError as e:
    print("LossFunction() ->", e)

# 2. gradient check: a derivative tem de bater certo com a derivada numérica da loss
#    derivada numérica de cada valor: (L(p + eps) - L(p - eps)) / (2 * eps)
def derivada_numerica(loss_fn, y_true, y_pred, eps=1e-6):
    numerica = np.zeros_like(y_pred)
    for i in range(y_pred.shape[0]):
        for j in range(y_pred.shape[1]):
            p_mais, p_menos = y_pred.copy(), y_pred.copy()
            p_mais[i, j] += eps
            p_menos[i, j] -= eps
            numerica[i, j] = (loss_fn.loss(y_true, p_mais) - loss_fn.loss(y_true, p_menos)) / (2 * eps)
    return numerica

np.random.seed(0)
y_true = np.random.randint(0, 2, size=(5, 1)).astype(float)  # rótulos 0/1
y_pred = np.random.uniform(0.05, 0.95, size=(5, 1))         # probabilidades longe de 0 e de 1

for loss_fn in [MeanSquaredError(), BinaryCrossEntropy()]:
    analitica = loss_fn.derivative(y_true, y_pred)
    numerica = derivada_numerica(loss_fn, y_true, y_pred)
    print(f"{type(loss_fn).__name__}: loss = {loss_fn.loss(y_true, y_pred):.4f} | "
          f"diferença máx. = {np.abs(analitica - numerica).max():.1e}")

# 3. a BCE castiga muito uma previsão confiante e errada
bce = BinaryCrossEntropy()
y = np.array([[1.0]])
print("y = 1, p = 0.9  ->", round(bce.loss(y, np.array([[0.9]])), 3))
print("y = 1, p = 0.01 ->", round(bce.loss(y, np.array([[0.01]])), 3))
print("y = 1, p = 0    ->", round(bce.loss(y, np.array([[0.0]])), 3), "(sem o clip dava inf)")

LossFunction() -> Can't instantiate abstract class LossFunction with abstract methods derivative, loss
MeanSquaredError: loss = 0.2969 | diferença máx. = 5.3e-11
BinaryCrossEntropy: loss = 4.0772 | diferença máx. = 5.1e-10
y = 1, p = 0.9  -> 0.105
y = 1, p = 0.01 -> 4.605
y = 1, p = 0    -> 34.539 (sem o clip dava inf)


**Conclusão (losses):**
- A `derivative` de cada loss bate certo com a derivada numérica (diferença ~1e-10): é mesmo a derivada da `loss`. É este valor que entra no backward pela última camada.
- MSE para regressão; BCE para classificação binária, com sigmoide na última camada.
- A BCE castiga muito os erros confiantes: com y = 1, p = 0.9 dá 0.105, mas p = 0.01 dá 4.6.
- O `np.clip` evita log(0) = -inf: com p = 0 a loss fica 34.5 em vez de inf.


## Optimizers (Slides 6-8)


In [ ]:
from si.neural_networks.optimizers import Optimizer, SGD

# classe base é abstrata
try:
    Optimizer(0.01)
except TypeError as e:
    print("Optimizer() ->", e)

# com momentum = 0, o SGD é o gradient descent normal: w - α * gradiente
w = np.array([[1.0, -2.0], [0.5, 3.0]])
grad = np.array([[0.2, -0.4], [1.0, 0.0]])
sgd = SGD(learning_rate=0.1)
print("Igual a w - α·grad:", np.allclose(sgd.update(w, grad), w - 0.1 * grad))
print("retained_gradient criado com a shape de w:", sgd.retained_gradient.shape)

# efeito do momentum: minimizar f(w) = w², cujo mínimo é w = 0 (gradiente = 2w)
# com α = 1.05 o passo é grande demais: o gradient descent salta por cima do mínimo
# e cada salto é maior do que o anterior (zigue-zague a crescer)
for beta in [0.0, 0.9]:
    opt = SGD(learning_rate=1.05, momentum=beta)
    w = np.array([[1.0]])
    valores = []
    for passo in range(30):
        w = opt.update(w, 2 * w)
        valores.append(round(float(w[0, 0]), 3))
    print(f"momentum = {beta}: primeiros passos {valores[:6]} ... passo 30: {valores[-1]}")
    # maior distância ao mínimo (|w|) em cada bloco de 10 passos: se cresce, diverge; se encolhe, converge
    amplitudes = [max(abs(v) for v in valores[i:i + 10]) for i in (0, 10, 20)]
    print(f"maior |w| nos passos 1-10, 11-20, 21-30: {amplitudes}")

Optimizer() -> Can't instantiate abstract class Optimizer with abstract method update
Igual a w - α·grad: True
retained_gradient criado com a shape de w: (2, 2)
momentum = 0.0: primeiros passos [-1.1, 1.21, -1.331, 1.464, -1.611, 1.772] ... passo 30: 17.449
maior |w| nos passos 1-10, 11-20, 21-30: [2.594, 6.727, 17.449]
momentum = 0.9: primeiros passos [0.79, 0.435, 0.024, -0.35, -0.614, -0.723] ... passo 30: -0.03
maior |w| nos passos 1-10, 11-20, 21-30: [0.79, 0.508, 0.273]


**Conclusão (optimizers):**
- Com `momentum = 0`, o `SGD` faz exatamente w − α·gradiente (gradient descent normal).
- O `retained_gradient` só é criado no primeiro `update`, com a shape dos parâmetros. Por isso, no `initialize` da `DenseLayer`, os pesos e os biases têm cada um a sua cópia do optimizer (`copy.deepcopy`).
- Em f(w) = w² com α = 1.05, sem momentum, cada passo salta por cima do mínimo e o salto cresce (w chega a 17.4 ao passo 30): diverge.
- Com momentum 0.9 também passa por cima do mínimo (chega a −0.72), mas cada oscilação é mais pequena do que a anterior (maior |w| por bloco de 10 passos: 0.79 → 0.51 → 0.27, contra 2.6 → 6.7 → 17.4 sem momentum). A média dos gradientes trava o zigue-zague e w aproxima-se do mínimo (−0.03 ao passo 30).

## NeuralNetwork (slides(9-13))


In [5]:
import numpy as np
from si.metrics.accuracy import accuracy
from si.io.csv_file import read_csv
from si.model_selection.split import stratified_train_test_split
from si.neural_networks.neural_network import NeuralNetwork
from si.neural_networks.layers import DenseLayer
from si.neural_networks.activation import ReLUActivation, SigmoidActivation
from si.neural_networks.optimizers import SGD
from si.neural_networks.losses import BinaryCrossEntropy

# accuracy: rótulos 1D como antes, e agora também probabilidades (n, 1) e (n, k)
y = np.array([0, 1, 1])
print("rótulos 1D:      ", round(accuracy(y, np.array([0, 1, 0])), 3))
print("prob. classe 1:  ", round(accuracy(y, np.array([[0.2], [0.9], [0.4]])), 3))
print("prob. por classe:", round(accuracy(y, np.array([[0.7, 0.3], [0.1, 0.9], [0.6, 0.4]])), 3))

# rede no breast-bin (9 features, classificação binária)
data = read_csv('../datasets/breast_bin/breast-bin.csv', sep=',', features=True, label=True)
train, test = stratified_train_test_split(data, test_size=0.3, random_state=42)

np.random.seed(42)
net = NeuralNetwork(epochs=20, batch_size=16, optimizer=SGD, learning_rate=0.01,
                    loss=BinaryCrossEntropy, metric=accuracy)
net.add(DenseLayer(4, (9,)))   # só a 1.ª camada precisa do input_shape
net.add(ReLUActivation())
net.add(DenseLayer(1))
net.add(SigmoidActivation())

# o add ligou as camadas: o input de cada uma é o output da anterior
for layer in net.layers:
    print(f"{layer.layer_name():18} input {layer.input_shape()} -> output {layer.output_shape()} | parâmetros: {layer.parameters()}")

net.fit(train)
for epoch in [1, 10, 20]:
    h = net.history[epoch]
    print(f"época {epoch:2}: loss = {h['loss']:.2f} | accuracy = {h['metric']:.3f}")

print("previsões:", net.predict(test).shape, "| accuracy no teste:", round(net.score(test), 3))

rótulos 1D:       0.667
prob. classe 1:   0.667
prob. por classe: 0.667
DenseLayer         input (9,) -> output (4,) | parâmetros: 40
ReLUActivation     input (4,) -> output (4,) | parâmetros: 0
DenseLayer         input (4,) -> output (1,) | parâmetros: 5
SigmoidActivation  input (1,) -> output (1,) | parâmetros: 0
época  1: loss = 235.63 | accuracy = 0.820
época 10: loss = 66.36 | accuracy = 0.955
época 20: loss = 58.34 | accuracy = 0.965
previsões: (209, 1) | accuracy no teste: 0.967


**Conclusão (NeuralNetwork):**
- O `add` liga as camadas sozinho: só a 1.ª precisa do `input_shape`. Parâmetros: 9·4 + 4 = 40 e 4·1 + 1 = 5.
- Em cada mini-batch: forward → derivada da loss → backward. Os pesos mudam depois de cada mini-batch (com batch_size = 16, são ~31 atualizações por época), e não só uma vez por época.
- A rede aprende: no treino, a loss desce de 235.6 para 58.3 e a accuracy sobe de 0.82 para 0.965. A BCE é uma soma, por isso o valor depende do número de amostras (≈ 0.12 por amostra no fim).
- No teste, com dados que a rede nunca viu, a accuracy é 0.967.
- A `accuracy` passou a aceitar as probabilidades da rede: (n, 1) arredonda e (n, k) usa argmax. Os rótulos 1D ficam iguais, por isso os modelos das outras aulas não mudam.